In [ ]:
# ============================================================
# Ensemble Method — STACKING v3 (Expanded)
# Improvements over v2:
#   1. More base models: RF, ExtraTrees, MLP, KNN, GNB, QDA,
#      Bagging, AdaBoost, SGD (all now ACTIVE)
#   2. More meta-learner options: XGBoost, CatBoost, LightGBM,
#      LogReg, Ridge, ElasticNet, MLP, RandomForest, ExtraTrees,
#      SVM, GradientBoosting, KNN
#   3. Combined MCC + AUROC objective (0.6 / 0.4 weighted)
#   4. Honest nested-CV threshold evaluation
# ============================================================

import numpy as np
import pandas as pd
import optuna
import warnings

from sklearn.naive_bayes           import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble              import (
    BaggingClassifier, AdaBoostClassifier,
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier,
)
from sklearn.linear_model          import (
    SGDClassifier, LogisticRegression,
    RidgeClassifier, ElasticNet,
)
from sklearn.svm                   import SVC
from sklearn.neural_network        import MLPClassifier
from sklearn.neighbors             import KNeighborsClassifier
from sklearn.preprocessing         import StandardScaler
from sklearn.pipeline              import Pipeline
from sklearn.calibration           import CalibratedClassifierCV
from sklearn.model_selection       import StratifiedKFold, cross_val_predict
from sklearn.metrics               import roc_auc_score, matthews_corrcoef

import xgboost  as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:

# ────────────────────────────────────────────────────────────
# 1. Load Data
# ────────────────────────────────────────────────────────────
BASE = r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\ranked rf"

def load_numeric_features(filepath):
    df = pd.read_csv(filepath, low_memory=False)
    return df.select_dtypes(include=[np.number]).values.astype(float)

X_train = load_numeric_features(f"{BASE}\\X_train_rF_DEG_train_lung.csv")
X_test  = load_numeric_features(f"{BASE}\\X_test_rF_DEG_train_lung.csv")
y_train = pd.read_csv(f"{BASE}\\y_train_rF_DEG_train.csv", low_memory=False)["cancer_type"].values.astype(int)
y_test  = pd.read_csv(f"{BASE}\\y_test_rF_DEG_train.csv",  low_memory=False)["cancer_type"].values.astype(int)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")

CV       = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
CV_INNER = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)



In [ ]:

# ────────────────────────────────────────────────────────────
# 2. Build Base Models (all active, with probability calibration)
# ────────────────────────────────────────────────────────────
def calibrate(est):
    return CalibratedClassifierCV(est, method="isotonic", cv=3)


def build_base_models(trial):
    """
    Returns a dict of calibrated base models.
    ALL models are active (none commented out).
    """

    # ── Gradient Boosting family ────────────────────────────
    xgb_raw = xgb.XGBClassifier(
        n_estimators     = trial.suggest_int(  "xgb_n_est",  100, 400, step=50),
        learning_rate    = trial.suggest_float("xgb_lr",     0.01, 0.2,  log=True),
        max_depth        = trial.suggest_int(  "xgb_depth",  3, 9),
        subsample        = trial.suggest_float("xgb_sub",    0.5, 1.0),
        colsample_bytree = trial.suggest_float("xgb_col",    0.5, 1.0),
        reg_alpha        = trial.suggest_float("xgb_alpha",  1e-4, 10.0, log=True),
        reg_lambda       = trial.suggest_float("xgb_lambda", 1e-4, 10.0, log=True),
        eval_metric="logloss", random_state=42, n_jobs=-1,
    )

    lgb_raw = lgb.LGBMClassifier(
        n_estimators      = trial.suggest_int(  "lgb_n_est",     100, 400, step=50),
        learning_rate     = trial.suggest_float("lgb_lr",        0.01, 0.2,  log=True),
        max_depth         = trial.suggest_int(  "lgb_depth",     3, 9),
        num_leaves        = trial.suggest_int(  "lgb_leaves",    20, 150),
        subsample         = trial.suggest_float("lgb_sub",       0.5, 1.0),
        min_child_samples = trial.suggest_int(  "lgb_min_child", 5, 50),
        random_state=42, n_jobs=-1, verbose=-1,
    )

    hgb_raw = HistGradientBoostingClassifier(
        max_iter         = trial.suggest_int(  "hgb_iter",     100, 400, step=50),
        learning_rate    = trial.suggest_float("hgb_lr",       0.01, 0.2, log=True),
        max_depth        = trial.suggest_int(  "hgb_depth",    3, 9),
        min_samples_leaf = trial.suggest_int(  "hgb_min_leaf", 5, 50),
        random_state=42,
    )

    cat_raw = CatBoostClassifier(
        iterations    = trial.suggest_int(  "cat_iter",  100, 400, step=50),
        learning_rate = trial.suggest_float("cat_lr",    0.01, 0.2, log=True),
        depth         = trial.suggest_int(  "cat_depth", 3, 9),
        l2_leaf_reg   = trial.suggest_float("cat_l2",    1e-4, 10.0, log=True),
        random_seed=42, verbose=0,
    )

    # sklearn GradientBoosting (different internals from HGB)
    gb_raw = GradientBoostingClassifier(
        n_estimators  = trial.suggest_int(  "gb_n_est",   100, 300, step=50),
        learning_rate = trial.suggest_float("gb_lr",      0.01, 0.2, log=True),
        max_depth     = trial.suggest_int(  "gb_depth",   3, 7),
        subsample     = trial.suggest_float("gb_sub",     0.5, 1.0),
        random_state=42,
    )

    # ── Tree ensembles ──────────────────────────────────────
    rf_raw = RandomForestClassifier(
        n_estimators      = trial.suggest_int(  "rf_n_est",    100, 400, step=50),
        max_depth         = trial.suggest_int(  "rf_depth",    3, 20),
        min_samples_split = trial.suggest_int(  "rf_min_split",2, 20),
        min_samples_leaf  = trial.suggest_int(  "rf_min_leaf", 1, 10),
        max_features      = trial.suggest_categorical("rf_feat", ["sqrt", "log2"]),
        random_state=42, n_jobs=-1,
    )

    et_raw = ExtraTreesClassifier(
        n_estimators      = trial.suggest_int(  "et_n_est",    100, 400, step=50),
        max_depth         = trial.suggest_int(  "et_depth",    3, 20),
        min_samples_split = trial.suggest_int(  "et_min_split",2, 20),
        min_samples_leaf  = trial.suggest_int(  "et_min_leaf", 1, 10),
        max_features      = trial.suggest_categorical("et_feat", ["sqrt", "log2"]),
        random_state=42, n_jobs=-1,
    )

    bag_raw = BaggingClassifier(
        n_estimators = trial.suggest_int(  "bag_n_est", 50, 200, step=50),
        max_samples  = trial.suggest_float("bag_samp",  0.5, 1.0),
        max_features = trial.suggest_float("bag_feat",  0.5, 1.0),
        random_state=42, n_jobs=-1,
    )

    ada_raw = AdaBoostClassifier(
        n_estimators  = trial.suggest_int(  "ada_n_est", 50, 200, step=50),
        learning_rate = trial.suggest_float("ada_lr",    0.01, 1.0, log=True),
        random_state=42,
    )

    # ── Linear / kernel ────────────────────────────────────
    lr_raw = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C      = trial.suggest_float("lr_C",     1e-3, 100.0, log=True),
            solver = trial.suggest_categorical("lr_solver", ["lbfgs", "saga"]),
            max_iter=2000, random_state=42, n_jobs=-1,
        )),
    ])

    svm_raw = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            C     = trial.suggest_float("svm_C",     1e-2, 100.0, log=True),
            gamma = trial.suggest_float("svm_gamma", 1e-4, 1.0,   log=True),
            kernel="rbf", probability=True, random_state=42,
        )),
    ])

    sgd_raw = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SGDClassifier(
            loss  = trial.suggest_categorical("sgd_loss", ["log_loss", "modified_huber"]),
            alpha = trial.suggest_float("sgd_alpha", 1e-5, 1e-1, log=True),
            max_iter=1000, random_state=42, n_jobs=-1,
        )),
    ])

    # ── Neural network ──────────────────────────────────────
    mlp_raw = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(
            hidden_layer_sizes = trial.suggest_categorical(
                "mlp_layers", [(128, 64), (256, 128), (128, 64, 32), (64, 32)]),
            activation         = trial.suggest_categorical("mlp_act", ["relu", "tanh"]),
            alpha              = trial.suggest_float("mlp_alpha", 1e-5, 1e-1, log=True),
            learning_rate_init = trial.suggest_float("mlp_lr",   1e-4, 1e-2, log=True),
            max_iter=500, random_state=42, early_stopping=True,
        )),
    ])

    # ── Instance-based ─────────────────────────────────────
    knn_raw = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(
            n_neighbors = trial.suggest_int("knn_k", 3, 21, step=2),
            weights     = trial.suggest_categorical("knn_w", ["uniform", "distance"]),
            metric      = trial.suggest_categorical("knn_m", ["euclidean", "cosine"]),
            n_jobs=-1,
        )),
    ])

    # ── Probabilistic ──────────────────────────────────────
    gnb_raw = GaussianNB(
        var_smoothing = trial.suggest_float("gnb_smooth", 1e-10, 1e-5, log=True),
    )

    qda_raw = QuadraticDiscriminantAnalysis(
        reg_param = trial.suggest_float("qda_reg", 0.0, 0.5),
    )

    return {
        # Gradient boosting
        "XGBoost"            : calibrate(xgb_raw),
        "LightGBM"           : calibrate(lgb_raw),
        "HistGradBoost"      : calibrate(hgb_raw),
        "CatBoost"           : calibrate(cat_raw),
        "GradientBoosting"   : calibrate(gb_raw),
        # Tree ensembles
        "RandomForest"       : calibrate(rf_raw),
        "ExtraTrees"         : calibrate(et_raw),
        "Bagging"            : calibrate(bag_raw),
        "AdaBoost"           : calibrate(ada_raw),
        # Linear / kernel
        "LogisticRegression" : calibrate(lr_raw),
        "SVM_RBF"            : calibrate(svm_raw),
        "SGD"                : calibrate(sgd_raw),
        # Neural / instance
        "MLP"                : calibrate(mlp_raw),
        "KNN"                : calibrate(knn_raw),
        # Probabilistic
        "GaussianNB"         : calibrate(gnb_raw),
        "QDA"                : calibrate(qda_raw),
    }



In [ ]:

# ────────────────────────────────────────────────────────────
# 3. Build Meta-Learner  (12 options)
# ────────────────────────────────────────────────────────────
META_TYPES = [
    "xgboost",          # XGBClassifier
    "lightgbm",         # LGBMClassifier
    "catboost",         # CatBoostClassifier
    "gradboost",        # sklearn GradientBoostingClassifier
    "randomforest",     # RandomForestClassifier
    "extratrees",       # ExtraTreesClassifier
    "logreg",           # LogisticRegression
    "ridge",            # RidgeClassifier (calibrated)
    "elasticnet",       # ElasticNet wrapped in calibration
    "svm",              # SVC with RBF kernel
    "mlp",              # MLPClassifier
    "knn",              # KNeighborsClassifier
]


def build_meta_learner(trial):
    meta_type = trial.suggest_categorical("meta_type", META_TYPES)

    # ── Gradient boosting meta-learners ────────────────────
    if meta_type == "xgboost":
        return xgb.XGBClassifier(
            n_estimators  = trial.suggest_int(  "meta_xgb_n_est", 50, 300, step=50),
            learning_rate = trial.suggest_float("meta_xgb_lr",    0.01, 0.3, log=True),
            max_depth     = trial.suggest_int(  "meta_xgb_depth", 2, 6),
            subsample     = trial.suggest_float("meta_xgb_sub",   0.6, 1.0),
            eval_metric="logloss", random_state=42, n_jobs=-1,
        )

    elif meta_type == "lightgbm":
        return lgb.LGBMClassifier(
            n_estimators      = trial.suggest_int(  "meta_lgb_n_est",  50, 300, step=50),
            learning_rate     = trial.suggest_float("meta_lgb_lr",     0.01, 0.3, log=True),
            max_depth         = trial.suggest_int(  "meta_lgb_depth",  2, 6),
            num_leaves        = trial.suggest_int(  "meta_lgb_leaves", 10, 60),
            min_child_samples = trial.suggest_int(  "meta_lgb_min",    5, 30),
            random_state=42, n_jobs=-1, verbose=-1,
        )

    elif meta_type == "catboost":
        return CatBoostClassifier(
            iterations    = trial.suggest_int(  "meta_cat_iter",  50, 300, step=50),
            learning_rate = trial.suggest_float("meta_cat_lr",    0.01, 0.3, log=True),
            depth         = trial.suggest_int(  "meta_cat_depth", 2, 6),
            l2_leaf_reg   = trial.suggest_float("meta_cat_l2",    1e-3, 10.0, log=True),
            random_seed=42, verbose=0,
        )

    elif meta_type == "gradboost":
        return GradientBoostingClassifier(
            n_estimators  = trial.suggest_int(  "meta_gb_n_est", 50, 200, step=50),
            learning_rate = trial.suggest_float("meta_gb_lr",    0.01, 0.3, log=True),
            max_depth     = trial.suggest_int(  "meta_gb_depth", 2, 5),
            subsample     = trial.suggest_float("meta_gb_sub",   0.6, 1.0),
            random_state=42,
        )

    # ── Tree ensemble meta-learners ────────────────────────
    elif meta_type == "randomforest":
        return RandomForestClassifier(
            n_estimators      = trial.suggest_int(  "meta_rf_n_est",    50, 300, step=50),
            max_depth         = trial.suggest_int(  "meta_rf_depth",    2, 10),
            min_samples_leaf  = trial.suggest_int(  "meta_rf_min_leaf", 1, 10),
            max_features      = trial.suggest_categorical("meta_rf_feat", ["sqrt", "log2"]),
            random_state=42, n_jobs=-1,
        )

    elif meta_type == "extratrees":
        return ExtraTreesClassifier(
            n_estimators      = trial.suggest_int(  "meta_et_n_est",    50, 300, step=50),
            max_depth         = trial.suggest_int(  "meta_et_depth",    2, 10),
            min_samples_leaf  = trial.suggest_int(  "meta_et_min_leaf", 1, 10),
            max_features      = trial.suggest_categorical("meta_et_feat", ["sqrt", "log2"]),
            random_state=42, n_jobs=-1,
        )

    # ── Linear meta-learners ───────────────────────────────
    elif meta_type == "logreg":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C       = trial.suggest_float("meta_lr_C",      1e-3, 100.0, log=True),
                penalty = trial.suggest_categorical("meta_lr_penalty", ["l1", "l2"]),
                solver  = "saga",
                max_iter=2000, random_state=42,
            )),
        ])

    elif meta_type == "ridge":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", CalibratedClassifierCV(
                RidgeClassifier(
                    alpha=trial.suggest_float("meta_ridge_alpha", 1e-2, 100.0, log=True)
                ),
                method="isotonic", cv=3,
            )),
        ])

    elif meta_type == "elasticnet":
        # ElasticNet is a regressor — wrap with calibration for classification
        l1_ratio = trial.suggest_float("meta_en_l1ratio", 0.0, 1.0)
        alpha    = trial.suggest_float("meta_en_alpha",   1e-4, 10.0, log=True)
        # Use SGDClassifier with elasticnet penalty (native classifier)
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", CalibratedClassifierCV(
                SGDClassifier(
                    loss="modified_huber",
                    penalty="elasticnet",
                    l1_ratio=l1_ratio,
                    alpha=alpha,
                    max_iter=1000, random_state=42,
                ),
                method="isotonic", cv=3,
            )),
        ])

    # ── Kernel / neural / instance meta-learners ──────────
    elif meta_type == "svm":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(
                C      = trial.suggest_float("meta_svm_C",     1e-2, 100.0, log=True),
                gamma  = trial.suggest_float("meta_svm_gamma", 1e-4, 1.0,   log=True),
                kernel = trial.suggest_categorical("meta_svm_kernel", ["rbf", "poly", "sigmoid"]),
                probability=True, random_state=42,
            )),
        ])

    elif meta_type == "mlp":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes = trial.suggest_categorical(
                    "meta_mlp_layers", [(64,), (128, 64), (64, 32), (32, 16)]),
                activation         = trial.suggest_categorical("meta_mlp_act", ["relu", "tanh"]),
                alpha              = trial.suggest_float("meta_mlp_alpha", 1e-5, 1e-1, log=True),
                learning_rate_init = trial.suggest_float("meta_mlp_lr",   1e-4, 1e-2, log=True),
                max_iter=500, random_state=42, early_stopping=True,
            )),
        ])

    elif meta_type == "knn":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(
                n_neighbors = trial.suggest_int("meta_knn_k", 3, 15, step=2),
                weights     = trial.suggest_categorical("meta_knn_w", ["uniform", "distance"]),
                metric      = trial.suggest_categorical("meta_knn_m", ["euclidean", "cosine"]),
                n_jobs=-1,
            )),
        ])

    else:
        raise ValueError(f"Unknown meta_type: {meta_type}")



In [ ]:

# ────────────────────────────────────────────────────────────
# 4. Combined Objective: 0.6 × MCC + 0.4 × AUROC
# ────────────────────────────────────────────────────────────
def combined_score(y_true, proba):
    best_mcc = max(
        matthews_corrcoef(y_true, (proba >= t).astype(int))
        for t in np.linspace(0.2, 0.8, 61)
    )
    auroc = roc_auc_score(y_true, proba)
    return 0.6 * best_mcc + 0.4 * auroc


# ────────────────────────────────────────────────────────────
# 5. Optuna Objective
# ────────────────────────────────────────────────────────────
def objective(trial):
    base_models  = build_base_models(trial)
    meta_learner = build_meta_learner(trial)
    n_base = len(base_models)

    oof_meta = np.zeros((X_train.shape[0], n_base))
    for i, (name, model) in enumerate(base_models.items()):
        oof_meta[:, i] = cross_val_predict(
            model, X_train, y_train, cv=CV, method="predict_proba"
        )[:, 1]

    meta_oof = cross_val_predict(
        meta_learner, oof_meta, y_train, cv=CV, method="predict_proba"
    )[:, 1]

    return combined_score(y_train, meta_oof)


# ────────────────────────────────────────────────────────────
# 6. Run Optimization
# ────────────────────────────────────────────────────────────
print("Starting Stacking Ensemble Optimization (v3)...")
print(f"  Base models   : 16 active")
print(f"  Meta-learners : {len(META_TYPES)} options → {META_TYPES}")

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n🏆 Best OOF Combined Score (0.6×MCC + 0.4×AUROC): {study.best_value:.4f}")
print(f"   Meta-learner chosen : {study.best_params['meta_type']}")
print("Best Parameters:")
for k, v in sorted(study.best_params.items()):
    print(f"  {k:40s} = {v}")



In [ ]:

# ────────────────────────────────────────────────────────────
# 7. Rebuild Final Models from Best Params
# ────────────────────────────────────────────────────────────
best = study.best_params

FINAL_BASE = {
    "XGBoost": calibrate(xgb.XGBClassifier(
        n_estimators=best["xgb_n_est"], learning_rate=best["xgb_lr"],
        max_depth=best["xgb_depth"], subsample=best["xgb_sub"],
        colsample_bytree=best["xgb_col"], reg_alpha=best["xgb_alpha"],
        reg_lambda=best["xgb_lambda"], eval_metric="logloss",
        random_state=42, n_jobs=-1,
    )),
    # "LightGBM": calibrate(lgb.LGBMClassifier(
    #     n_estimators=best["lgb_n_est"], learning_rate=best["lgb_lr"],
    #     max_depth=best["lgb_depth"], num_leaves=best["lgb_leaves"],
    #     subsample=best["lgb_sub"], min_child_samples=best["lgb_min_child"],
    #     random_state=42, n_jobs=-1, verbose=-1,
    # )),
    "HistGradBoost": calibrate(HistGradientBoostingClassifier(
        max_iter=best["hgb_iter"], learning_rate=best["hgb_lr"],
        max_depth=best["hgb_depth"], min_samples_leaf=best["hgb_min_leaf"],
        random_state=42,
    )),
    "CatBoost": calibrate(CatBoostClassifier(
        iterations=best["cat_iter"], learning_rate=best["cat_lr"],
        depth=best["cat_depth"], l2_leaf_reg=best["cat_l2"],
        random_seed=42, verbose=0,
    )),
    "GradientBoosting": calibrate(GradientBoostingClassifier(
        n_estimators=best["gb_n_est"], learning_rate=best["gb_lr"],
        max_depth=best["gb_depth"], subsample=best["gb_sub"],
        random_state=42,
    )),
    # "RandomForest": calibrate(RandomForestClassifier(
    #     n_estimators=best["rf_n_est"], max_depth=best["rf_depth"],
    #     min_samples_split=best["rf_min_split"], min_samples_leaf=best["rf_min_leaf"],
    #     max_features=best["rf_feat"], random_state=42, n_jobs=-1,
    # )),
    # "ExtraTrees": calibrate(ExtraTreesClassifier(
    #     n_estimators=best["et_n_est"], max_depth=best["et_depth"],
    #     min_samples_split=best["et_min_split"], min_samples_leaf=best["et_min_leaf"],
    #     max_features=best["et_feat"], random_state=42, n_jobs=-1,
    # )),
    # "Bagging": calibrate(BaggingClassifier(
    #     n_estimators=best["bag_n_est"], max_samples=best["bag_samp"],
    #     max_features=best["bag_feat"], random_state=42, n_jobs=-1,
    # )),
    # "AdaBoost": calibrate(AdaBoostClassifier(
    #     n_estimators=best["ada_n_est"], learning_rate=best["ada_lr"],
    #     random_state=42,
    # )),
    "LogisticRegression": calibrate(Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=best["lr_C"], solver=best["lr_solver"],
                                    max_iter=2000, random_state=42, n_jobs=-1)),
    ])),
    "SVM_RBF": calibrate(Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(C=best["svm_C"], gamma=best["svm_gamma"],
                    kernel="rbf", probability=True, random_state=42)),
    ])),
    "SGD": calibrate(Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SGDClassifier(loss=best["sgd_loss"], alpha=best["sgd_alpha"],
                               max_iter=1000, random_state=42, n_jobs=-1)),
    ])),
    # "MLP": calibrate(Pipeline([
    #     ("scaler", StandardScaler()),
    #     ("clf", MLPClassifier(
    #         hidden_layer_sizes=best["mlp_layers"], activation=best["mlp_act"],
    #         alpha=best["mlp_alpha"], learning_rate_init=best["mlp_lr"],
    #         max_iter=500, random_state=42, early_stopping=True,
    #     )),
    # ])),
    # "KNN": calibrate(Pipeline([
    #     ("scaler", StandardScaler()),
    #     ("clf", KNeighborsClassifier(
    #         n_neighbors=best["knn_k"], weights=best["knn_w"],
    #         metric=best["knn_m"], n_jobs=-1,
    #     )),
    # ])),
    # "GaussianNB": calibrate(GaussianNB(var_smoothing=best["gnb_smooth"])),
    # "QDA"       : calibrate(QuadraticDiscriminantAnalysis(reg_param=best["qda_reg"])),
}

# ── Final meta-learner (mirrors build_meta_learner logic) ───
mt = best["meta_type"]

if mt == "xgboost":
    FINAL_META = xgb.XGBClassifier(
        n_estimators=best["meta_xgb_n_est"], learning_rate=best["meta_xgb_lr"],
        max_depth=best["meta_xgb_depth"], subsample=best["meta_xgb_sub"],
        eval_metric="logloss", random_state=42, n_jobs=-1,
    )
elif mt == "lightgbm":
    FINAL_META = lgb.LGBMClassifier(
        n_estimators=best["meta_lgb_n_est"], learning_rate=best["meta_lgb_lr"],
        max_depth=best["meta_lgb_depth"], num_leaves=best["meta_lgb_leaves"],
        min_child_samples=best["meta_lgb_min"],
        random_state=42, n_jobs=-1, verbose=-1,
    )
elif mt == "catboost":
    FINAL_META = CatBoostClassifier(
        iterations=best["meta_cat_iter"], learning_rate=best["meta_cat_lr"],
        depth=best["meta_cat_depth"], l2_leaf_reg=best["meta_cat_l2"],
        random_seed=42, verbose=0,
    )
elif mt == "gradboost":
    FINAL_META = GradientBoostingClassifier(
        n_estimators=best["meta_gb_n_est"], learning_rate=best["meta_gb_lr"],
        max_depth=best["meta_gb_depth"], subsample=best["meta_gb_sub"],
        random_state=42,
    )
elif mt == "randomforest":
    FINAL_META = RandomForestClassifier(
        n_estimators=best["meta_rf_n_est"], max_depth=best["meta_rf_depth"],
        min_samples_leaf=best["meta_rf_min_leaf"], max_features=best["meta_rf_feat"],
        random_state=42, n_jobs=-1,
    )
elif mt == "extratrees":
    FINAL_META = ExtraTreesClassifier(
        n_estimators=best["meta_et_n_est"], max_depth=best["meta_et_depth"],
        min_samples_leaf=best["meta_et_min_leaf"], max_features=best["meta_et_feat"],
        random_state=42, n_jobs=-1,
    )
elif mt == "logreg":
    FINAL_META = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=best["meta_lr_C"], penalty=best["meta_lr_penalty"],
            solver="saga", max_iter=2000, random_state=42,
        )),
    ])
elif mt == "ridge":
    FINAL_META = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", CalibratedClassifierCV(
            RidgeClassifier(alpha=best["meta_ridge_alpha"]),
            method="isotonic", cv=3,
        )),
    ])
elif mt == "elasticnet":
    FINAL_META = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", CalibratedClassifierCV(
            SGDClassifier(
                loss="modified_huber", penalty="elasticnet",
                l1_ratio=best["meta_en_l1ratio"], alpha=best["meta_en_alpha"],
                max_iter=1000, random_state=42,
            ),
            method="isotonic", cv=3,
        )),
    ])
elif mt == "svm":
    FINAL_META = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            C=best["meta_svm_C"], gamma=best["meta_svm_gamma"],
            kernel=best["meta_svm_kernel"],
            probability=True, random_state=42,
        )),
    ])
elif mt == "mlp":
    FINAL_META = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(
            hidden_layer_sizes=best["meta_mlp_layers"], activation=best["meta_mlp_act"],
            alpha=best["meta_mlp_alpha"], learning_rate_init=best["meta_mlp_lr"],
            max_iter=500, random_state=42, early_stopping=True,
        )),
    ])
elif mt == "knn":
    FINAL_META = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(
            n_neighbors=best["meta_knn_k"], weights=best["meta_knn_w"],
            metric=best["meta_knn_m"], n_jobs=-1,
        )),
    ])
else:
    raise ValueError(f"Unknown meta_type: {mt}")



In [ ]:

# ────────────────────────────────────────────────────────────
# 8. Build Meta-Features, Train & Score Every Model
# ────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    matthews_corrcoef, confusion_matrix, classification_report,
)

def full_metrics(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "ACC" : accuracy_score(y_true, pred),
        "SEN" : tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        "SPE" : tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "F1"  : f1_score(y_true, pred, zero_division=0),
        "AUC" : roc_auc_score(y_true, proba),
        "MCC" : matthews_corrcoef(y_true, pred),
    }

def best_threshold_metrics(y_true, proba):
    best_t, best_mcc = 0.5, -1.0
    for t in np.linspace(0.1, 0.9, 81):
        mcc = matthews_corrcoef(y_true, (proba >= t).astype(int))
        if mcc > best_mcc:
            best_mcc, best_t = mcc, t
    return full_metrics(y_true, proba, threshold=best_t), best_t

n_base      = len(FINAL_BASE)
oof_meta_X  = np.zeros((X_train.shape[0], n_base))
test_meta_X = np.zeros((X_test.shape[0],  n_base))
model_oof_results  = {}
model_test_results = {}

HDR = f"{'Model':<25} {'ACC':>6} {'SEN':>6} {'SPE':>6} {'F1':>6} {'AUC':>6} {'MCC':>6}"
SEP = "-" * len(HDR)

print("\nBuilding meta-features & scoring each base model...")
print(SEP); print(HDR); print(SEP)

for i, (name, model) in enumerate(FINAL_BASE.items()):
    oof_proba = cross_val_predict(
        model, X_train, y_train, cv=CV, method="predict_proba"
    )[:, 1]
    oof_meta_X[:, i] = oof_proba

    m_oof, thr_oof = best_threshold_metrics(y_train, oof_proba)
    model_oof_results[name] = (m_oof, thr_oof)

    model.fit(X_train, y_train)
    test_proba = model.predict_proba(X_test)[:, 1]
    test_meta_X[:, i] = test_proba

    m_test = full_metrics(y_test, test_proba, threshold=thr_oof)
    model_test_results[name] = m_test

    print(f"{name:<25} "
          f"{m_oof['ACC']:>6.4f} {m_oof['SEN']:>6.4f} {m_oof['SPE']:>6.4f} "
          f"{m_oof['F1']:>6.4f} {m_oof['AUC']:>6.4f} {m_oof['MCC']:>6.4f}"
          f"   (thr={thr_oof:.2f})")

print(SEP)

# ── Meta-learner ─────────────────────────────────────────────
FINAL_META.fit(oof_meta_X, y_train)
meta_test_proba = FINAL_META.predict_proba(test_meta_X)[:, 1]

meta_oof_proba = cross_val_predict(
    FINAL_META, oof_meta_X, y_train, cv=CV_INNER, method="predict_proba"
)[:, 1]

best_thr, best_mcc_val = 0.5, -1.0
for t in np.linspace(0.1, 0.9, 81):
    mcc = matthews_corrcoef(y_train, (meta_oof_proba >= t).astype(int))
    if mcc > best_mcc_val:
        best_mcc_val, best_thr = mcc, t



In [ ]:

# ────────────────────────────────────────────────────────────
# 9. Final Results
# ────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  STACKING ENSEMBLE v3 — FINAL TEST RESULTS")
print("=" * 70)
print(f"  Meta-learner      : {best['meta_type']}")
print(f"  Decision threshold: {best_thr:.2f}")
print()

print(f"  {'Model':<25} {'ACC':>6} {'SEN':>6} {'SPE':>6} {'F1':>6} {'AUC':>6} {'MCC':>6}")
print("  " + "-" * 62)
for name, m in model_test_results.items():
    print(f"  {name:<25} "
          f"{m['ACC']:>6.4f} {m['SEN']:>6.4f} {m['SPE']:>6.4f} "
          f"{m['F1']:>6.4f} {m['AUC']:>6.4f} {m['MCC']:>6.4f}")

print("  " + "-" * 62)
ens_m = full_metrics(y_test, meta_test_proba, threshold=best_thr)
print(f"  {'>>> ENSEMBLE':<25} "
      f"{ens_m['ACC']:>6.4f} {ens_m['SEN']:>6.4f} {ens_m['SPE']:>6.4f} "
      f"{ens_m['F1']:>6.4f} {ens_m['AUC']:>6.4f} {ens_m['MCC']:>6.4f}")
print("=" * 70)

print("\n  Detailed classification report (ensemble):")
y_pred = (meta_test_proba >= best_thr).astype(int)
print(classification_report(y_test, y_pred, digits=4))

print("\n  Base model ranking by OOF AUC:")
ranked = sorted(model_oof_results.items(), key=lambda x: x[1][0]["AUC"], reverse=True)
for rank, (name, (m, thr)) in enumerate(ranked, 1):
    bar = "█" * int(m["AUC"] * 20)
    print(f"  {rank:2}. {name:<25} AUC={m['AUC']:.4f}  F1={m['F1']:.4f}  MCC={m['MCC']:.4f}  {bar}")